In [1]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 3, Finished, Available, Finished, False)

In [2]:
stream_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/dbo/stream_orders"
bronze_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/bronze/orders"

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 4, Finished, Available, Finished, False)

In [3]:
# Reading from eventstream
df_raw = spark.readStream.format("delta").load(stream_path)

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 5, Finished, Available, Finished, False)

In [4]:
# Adding metadata columns 
df_orders = (
    df_raw
        .withColumn("ingested_at", current_timestamp())
        .withColumn("source", lit("eventstream"))
)

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 6, Finished, Available, Finished, False)

In [5]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 7, Finished, Available, Finished, False)

DataFrame[]

In [6]:
# Write to Bronze table
bronze_checkpoint = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Files/_checkpoints/bronze_orders"

query = (
    df_orders.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("bronze.orders")
)
query.awaitTermination()

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 8, Finished, Available, Finished, False)

In [7]:
df_bronze = spark.read.format("delta").load(bronze_path)
print("Bronze count:", df_bronze.count())
display(df_bronze)

StatementMeta(, 3634a233-b845-46b9-9daf-d65cc8133ad9, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 17bfdf98-6918-4072-8d0c-2fa99676872e)